# 4. SAN-M：共享 Value 的注意力与时序记忆

**研究问题**：SAN-M 如何把输入相关的全局注意力和参数固定的局部时序滤波合在同一个模块中？共享的 Value 投影怎样获得两路梯度？

**方法**：用小维度 PyTorch 模块实现 $Q,K,V$ 投影、多头注意力、逐通道时序卷积和逐元素融合，再检查形状、局部/全局影响及梯度。

**验证依据与边界**：所有实验仅用合成张量，可核对计算结构，不证明 ASR 准确率或分支互补收益；本本是离线、等长、无 padding 的教学简化，不加载权重或数据。

来源：[分享对话](https://chatgpt.com/share/6afa6156-807c-83ec-a8da-c083b8f7457b)、[SAN-M 论文](https://www.isca-archive.org/interspeech_2020/gao20_interspeech.html)、[FunASR attention.py 固定提交](https://github.com/modelscope/FunASR/blob/c1e3018bf0b16ac10d40fe7cf902cafd624ee4dd/funasr/models/sanm/attention.py)。本本不声明与官方 checkpoint 数值等价。

## 整体架构图



$X[B,T,D] 	o Q,K,V[B,T,D]$；Attention 使用拆头后的 $Q,K,V$，记忆分支直接使用同一个 $V$；两路输出都是 $[B,T,D]$。

In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(17)
print("PyTorch:", torch.__version__)


## 1. 两条分支的公式与形状

设 $X\in\mathbb{R}^{B\times T\times D}$，$H$ 个头，$d=D/H$。$Q=XW_Q$、$K=XW_K$、$V=XW_V$ 均为 $[B,T,D]$；拆头后是 $[B,H,T,d]$。

注意力分支：$A=\operatorname{softmax}(QK^\top/\sqrt d)$ 为 $[B,H,T,T]$；$O_{att}=W_O(AV)$ 为 $[B,T,D]$。权重随当前输入变化，可关联远处帧。

记忆分支：$M(V)_{t,c}=v_{t,c}+\sum_{k=-L}^{R}w_{k,c}v_{t+k,c}$，超出序列部分补零。每个通道独立的 $w_{k,c}$ 在训练后对所有样本共享。这里用 `groups=D` 的 `Conv1d` 实现；卷积在 PyTorch 中按相关运算定义，卷积核索引从 $-L$ 到 $R$。

融合：$Y=O_{att}+M(V)$，形状仍为 $[B,T,D]$。注意这里的 $+V$ 属于记忆分支，不能与 Encoder block 的 $+X$ 混淆。

In [ ]:
class SANM(nn.Module):
    def __init__(self, dim=8, heads=2, left=2, right=2):
        super().__init__()
        if dim % heads:
            raise ValueError("dim 必须能被 heads 整除")
        if left < 0 or right < 0:
            raise ValueError("上下文长度必须非负")
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.left, self.right = left, right
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out_proj = nn.Linear(dim, dim)
        self.memory_conv = nn.Conv1d(dim, dim, left + right + 1,
                                     groups=dim, bias=False)

    def branches(self, x):
        B, T, D = x.shape
        if D != self.dim:
            raise ValueError("最后一维必须等于 dim")
        q, k, v = self.qkv(x).chunk(3, dim=-1)  # 各 [B,T,D]
        def split(z):
            return z.reshape(B, T, self.heads, self.head_dim).transpose(1, 2)
        qh, kh, vh = split(q), split(k), split(v)  # 各 [B,H,T,d]
        scores = qh @ kh.transpose(-2, -1) / math.sqrt(self.head_dim)
        weights = scores.softmax(dim=-1)          # [B,H,T,T]
        context = weights @ vh                     # [B,H,T,d]
        attention = self.out_proj(context.transpose(1, 2).reshape(B, T, D))
        padded = F.pad(v.transpose(1, 2), (self.left, self.right))
        memory = v + self.memory_conv(padded).transpose(1, 2)
        return attention, memory, v, weights

    def forward(self, x):
        attention, memory, _, _ = self.branches(x)
        return attention + memory


x = torch.randn(2, 7, 8, requires_grad=True)
sanm = SANM()
attention, memory, v, weights = sanm.branches(x)
y = sanm(x)
for name, tensor in [("X", x), ("V", v), ("A", weights),
                     ("Attention", attention), ("Memory", memory), ("Y", y)]:
    print(name, tuple(tensor.shape))
assert weights.shape == (2, 2, 7, 7)
assert y.shape == attention.shape == memory.shape == x.shape
assert torch.allclose(y, attention + memory)


## 2. 验证记忆分支的局部性

设 `left=right=2`，只改变第 3 帧的 $V$，那么卷积分量只能影响第 1 至 5 帧。以下先直接操作 $V$，隔离投影和注意力的作用。

In [ ]:
with torch.no_grad():
    sanm.memory_conv.weight.fill_(1.0)
    base = torch.zeros(1, 7, 8)
    impulse = base.clone()
    impulse[:, 3, 0] = 1.0
    def memory_from_v(v):
        z = F.pad(v.transpose(1, 2), (sanm.left, sanm.right))
        return v + sanm.memory_conv(z).transpose(1, 2)
    delta = memory_from_v(impulse) - memory_from_v(base)
    affected = torch.where(delta[0, :, 0] != 0)[0].tolist()
    print("受影响帧:", affected)
    assert affected == [1, 2, 3, 4, 5]
    assert torch.count_nonzero(delta[0, :, 1:]) == 0


## 3. 验证注意力的远程关联与联合梯度

Softmax 前的得分依赖输入，权重矩阵覆盖整个 $T	imes T$。梯度分别从注意力和记忆分支回到共享的 Value 投影；下面用同一模块分别反传，检查两条路径均非零。非零梯度只证明计算图连通，不能证明训练后互补。

In [ ]:
with torch.no_grad():
    print("第 0 帧对末帧的注意力权重:", weights[0, :, 0, -1].tolist())
assert torch.all(weights[0, :, 0, -1] > 0)

for branch in ("attention", "memory", "combined"):
    sanm.zero_grad(set_to_none=True)
    a, m, _, _ = sanm.branches(x.detach())
    output = {"attention": a, "memory": m, "combined": a + m}[branch]
    output.square().mean().backward()
    # qkv 的最后 D 行对应共享 V 投影。
    value_grad = sanm.qkv.weight.grad[2 * sanm.dim:].norm().item()
    print(branch, "V 梯度:", value_grad)
    assert value_grad > 0
    if branch == "combined":
        assert sanm.out_proj.weight.grad.norm() > 0
        assert sanm.memory_conv.weight.grad.norm() > 0


## 4. Encoder block 与实验边界

一个教学版 block 可写成 $Z=X+\operatorname{SANM}(\operatorname{LN}(X))$，$O=Z+\operatorname{FFN}(\operatorname{LN}(Z))$。这层残差与记忆分支内的 $+V$ 是两次不同的相加。生产实现还会处理 padding、mask、shift、缓存、dropout 等，不能直接载入 FunASR 权重。若要验证识别收益，需在同数据、同训练预算下比较 Attention、Memory、二者融合的 CER/WER；这里的合成实验不提供该证据。

In [ ]:
class SANMEncoderBlock(nn.Module):
    def __init__(self, dim=8, heads=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.sanm = SANM(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, 4 * dim), nn.ReLU(),
                                 nn.Linear(4 * dim, dim))

    def forward(self, x):
        z = x + self.sanm(self.norm1(x))
        return z + self.ffn(self.norm2(z))


block = SANMEncoderBlock()
out = block(torch.randn(2, 7, 8))
print("Encoder block 输出:", tuple(out.shape))
assert out.shape == (2, 7, 8)
